# 🚀 ViT AI Image Detector 训练

**运行前确保：** Runtime → Change runtime type → **A100 GPU**

预计训练时间: **30-45 分钟** (50K图片, 5 epochs)

## 1️⃣ 检查 GPU

In [ ]:
!nvidia-smi

## 2️⃣ 安装依赖

In [ ]:
!pip install -q torch torchvision transformers scikit-learn tqdm pillow kagglehub mlflow huggingface_hub

## 3️⃣ 克隆仓库

In [ ]:
# 克隆你的仓库
!git clone https://github.com/wanikua/ItsNotAI-Model-Training.git
%cd ItsNotAI-Model-Training

print('✅ 仓库克隆成功!')

In [ ]:
# 添加项目到 Python 路径
import sys
sys.path.insert(0, '/content/ItsNotAI-Model-Training')

# 验证导入
from src.models.vit_detector import ViTDetector
print('✅ 项目导入成功!')

## 4️⃣ 下载数据集

In [ ]:
from src.dataset.download_datasets import prepare_combined_dataset, verify_dataset

DATA_ROOT = '/content/data'
prepare_combined_dataset(DATA_ROOT)
verify_dataset(DATA_ROOT)

## 5️⃣ 开始训练! 🎯

In [ ]:
from src.training.config import TrainingConfig
from src.training.train_vit import Trainer

# A100 优化配置
config = TrainingConfig.for_colab_a100()
config.data_root = '/content/data'
config.output_dir = '/content/outputs'
config.num_epochs = 5
config.batch_size = 128  # A100 可以用大 batch
config.learning_rate = 2e-5
config.use_mlflow = False  # Colab 里禁用 MLflow

print('配置:')
print(f'  Batch size: {config.batch_size}')
print(f'  Epochs: {config.num_epochs}')
print(f'  Learning rate: {config.learning_rate}')

In [ ]:
# 开始训练!
trainer = Trainer(config)
test_metrics = trainer.train()

print('\n🎉 训练完成!')
print(f'测试集指标: {test_metrics}')

## 6️⃣ 保存模型到 Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 保存模型
!mkdir -p "/content/drive/MyDrive/AI-Models"
!cp -r /content/outputs/best_model "/content/drive/MyDrive/AI-Models/vit-ai-detector"

print('✅ 模型已保存到 Google Drive: AI-Models/vit-ai-detector')

## 7️⃣ 测试模型

In [ ]:
from src.models.vit_detector import ViTDetector
from PIL import Image
import requests
from io import BytesIO

# 加载训练好的模型
model = ViTDetector.load('/content/outputs/best_model')

# 测试一张图片
def test_url(url):
    response = requests.get(url)
    img = Image.open(BytesIO(response.content)).convert('RGB')
    result = model.predict(img)
    print(f'Prediction: {result.label}')
    print(f'Probabilities: real={result.probs[0]:.2%}, fake={result.probs[1]:.2%}')
    return result

# 替换为你想测试的图片 URL
# test_url('https://example.com/test_image.jpg')

In [ ]:
# 上传本地图片测试
from google.colab import files

print('上传一张图片测试:')
uploaded = files.upload()

for filename in uploaded.keys():
    img = Image.open(filename).convert('RGB')
    result = model.predict(img)
    print(f'\n文件: {filename}')
    print(f'  预测: {result.label}')
    print(f'  置信度: {max(result.probs):.2%}')